# Variational quantum classifier: parameter-shift rule và so sánh trung thực

Bạn tự viết **parameter-shift rule** (cách lấy gradient của mạch lượng tử chạy được cả trên phần cứng thật), dựng
một **variational quantum circuit** 2 qubit bằng PennyLane có data re-uploading, train nó trên `make_moons`, rồi
so với logistic regression, SVM RBF và hai MLP **trên cùng 5 lần chia dữ liệu**. Mục 5 lý thuyết có công thức
và bảng kết quả bạn sẽ tái hiện.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) chạy thí nghiệm, vẽ; test bỏ qua chúng.
- Dữ liệu `make_moons` sinh bằng scikit-learn, không cần tải.
- Chấm: `uv run pytest tracks/ai-engineer/modules/22-quantum-ml/exercises/intermediate/01-variational-classifier -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy
import time
import warnings

import numpy as np
import pennylane as qml

warnings.filterwarnings("ignore", category=UserWarning)

## 1. Parameter-shift rule

Với gate dạng $$e^{-i\theta P/2}$$ ($$P$$ là tích Pauli: RX, RY, RZ đều thế), mọi kỳ vọng đo được là hàm
$$f(\theta) = a + b\cos\theta + c\sin\theta$$ theo **từng** tham số. Từ đó (mục 5 lý thuyết):

$$
\frac{\partial f}{\partial \theta_i} = \frac{f(\theta + s\,e_i) - f(\theta - s\,e_i)}{2\sin s}
$$

đúng **chính xác** với mọi $$s$$ không là bội của $$\pi$$; mặc định $$s = \pi/2$$ cho mẫu số bằng 2. Khác sai phân hữu
hạn: không có sai số cắt cụt, và hai điểm đo cách xa nhau nên chịu được nhiễu shot.

Viết `parameter_shift_jacobian(f, params, shift=np.pi/2)`:
- `params` là mảng float shape bất kỳ; `f(params)` trả một số hoặc mảng shape `out` (ví dụ `(B,)`: một kỳ vọng
  cho mỗi mẫu).
- Trả mảng shape `params.shape + out`: phần tử `[idx]` là đạo hàm của toàn bộ output theo `params[idx]`.
- Mỗi tham số gọi `f` đúng hai lần (dịch $$+s$$ và $$-s$$, chỉ đúng một phần tử); được gọi thêm một lần ở điểm gốc
  để biết shape `out`. Không sửa `params` truyền vào.

In [ ]:
def parameter_shift_jacobian(f, params: np.ndarray, shift: float = np.pi / 2) -> np.ndarray:
    """Jacobian của f theo params bằng parameter-shift rule: (f(θ + s e_i) − f(θ − s e_i)) / (2 sin s).

    Trả mảng shape params.shape + np.shape(f(params)). Không sửa params.
    """
    base = np.array(params, dtype=float)              # bản sao: không đụng vào params của người gọi
    out_shape = np.shape(f(base))
    jac = np.zeros(base.shape + out_shape)
    for idx in np.ndindex(base.shape):
        shifted = base.copy()
        shifted[idx] = base[idx] + shift
        f_plus = np.asarray(f(shifted), dtype=float)
        shifted[idx] = base[idx] - shift
        f_minus = np.asarray(f(shifted), dtype=float)
        jac[idx] = (f_plus - f_minus) / (2 * np.sin(shift))
    return jac

In [ ]:
# explore: một qubit, f(θ) = <Z> sau RY(θ) = cos θ. So parameter-shift với sai phân hữu hạn, có và không có nhiễu shot
dev1 = qml.device("default.qubit", wires=1, seed=0)


@qml.qnode(dev1)
def f1(theta):
    qml.RY(theta[0], wires=0)
    return qml.expval(qml.PauliZ(0))


theta0 = np.array([0.8])
exact = -np.sin(0.8)
ps = parameter_shift_jacobian(lambda t: float(f1(t)), theta0)[0]
print(f"đạo hàm đúng {exact:.10f} | parameter-shift {ps:.10f} | lệch {abs(ps - exact):.1e}")
for eps in (1e-1, 1e-4, 1e-8):
    fd = (float(f1(theta0 + eps)) - float(f1(theta0 - eps))) / (2 * eps)
    print(f"sai phân trung tâm eps={eps:.0e}: {fd:.10f} | lệch {abs(fd - exact):.1e}")

# Trên phần cứng thật, mỗi kỳ vọng chỉ ước lượng được từ shots: lặp 200 lần ước lượng đạo hàm, 1000 shots mỗi lần đo
f1_shots = qml.set_shots(f1, shots=1000)
ps_runs = [parameter_shift_jacobian(lambda t: float(f1_shots(t)), theta0)[0] for _ in range(200)]
fd_runs = [(float(f1_shots(theta0 + 1e-2)) - float(f1_shots(theta0 - 1e-2))) / 2e-2 for _ in range(200)]
print(f"1000 shots, 200 lần: parameter-shift trung bình {np.mean(ps_runs):+.3f} std {np.std(ps_runs):.3f}"
      f" | sai phân eps=1e-2 trung bình {np.mean(fd_runs):+.3f} std {np.std(fd_runs):.3f}")

## 2. Mạch variational 2 qubit với data re-uploading

Input $$x = (x_0, x_1)$$ đã scale về $$[0, \pi]$$. Tham số `params` shape `(L, 2, 2)`. Mỗi layer $$l = 0..L-1$$:

1. Với wire $$w = 0, 1$$ theo thứ tự: `RY(x[w])`, rồi `RY(params[l, w, 0])`, rồi `RZ(params[l, w, 1])` trên wire $$w$$.
2. `CNOT` với control wire 0, target wire 1.

Output là $$\langle Z_0 \rangle \in [-1, 1]$$. Input được "nạp lại" ở mọi layer (**data re-uploading**): một lần nạp
chỉ cho hàm bậc 1 theo $$x$$ (một tần số), $$L$$ lần nạp cho chuỗi Fourier tới tần số $$L$$.

`make_vqc(n_layers)` trả một hàm `model(params, X)`: `X` shape `(B, 2)`, trả `np.ndarray` float shape `(B,)`. Dùng
`qml.device("default.qubit", wires=2)` và **parameter broadcasting** của PennyLane: truyền cả cột `X[:, w]` vào
`qml.RY` thì một lần chạy QNode tính cho cả batch, nhanh hơn vòng lặp Python hàng chục lần. Gắn chính QNode đó vào
thuộc tính `model.circuit` (`model.circuit = circuit`): cell `explore` ở mục 5 dùng nó để lấy gradient bằng autograd của PennyLane.

In [ ]:
def make_vqc(n_layers: int):
    """Trả model(params, X) -> np.ndarray shape (B,): <Z_0> của mạch re-uploading L layer, params shape (L, 2, 2)."""
    dev = qml.device("default.qubit", wires=2)

    @qml.qnode(dev)
    def circuit(params, X):
        for l in range(n_layers):
            for w in range(2):
                qml.RY(X[:, w], wires=w)               # broadcasting: một lần chạy cho cả batch
                qml.RY(params[l, w, 0], wires=w)
                qml.RZ(params[l, w, 1], wires=w)
            qml.CNOT(wires=[0, 1])
        return qml.expval(qml.PauliZ(0))

    def model(params, X):
        X = np.asarray(X, dtype=float)
        return np.asarray(circuit(np.asarray(params, dtype=float), X), dtype=float).reshape(len(X))

    model.circuit = circuit                            # QNode gốc, để cell explore lấy gradient bằng autograd
    return model

## 3. Loss, gradient và vòng train

Nhãn $$y \in \{-1, +1\}$$ (lớp 1 là $$+1$$), dự đoán lớp 1 khi output $$> 0$$. Loss là MSE

$$
\mathcal{L}(\theta) = \frac{1}{B}\sum_{b=1}^{B} \big(f_\theta(x_b) - y_b\big)^2, \qquad
\frac{\partial \mathcal{L}}{\partial \theta_i} = \frac{2}{B}\sum_b \big(f_\theta(x_b) - y_b\big)\frac{\partial f_\theta(x_b)}{\partial \theta_i}
$$

- `loss_and_grad(model, params, X, y_pm)` trả `(loss, grad)`: `loss` là float Python, `grad` cùng shape `params`,
  tính bằng **parameter_shift_jacobian của bạn** (Jacobian của output theo tham số) rồi chain rule ở trên.
- `train_vqc(model, X, y_pm, n_layers, steps=100, lr=0.3, seed=0)`: khởi tạo
  `params = np.random.default_rng(seed).uniform(-np.pi, np.pi, size=(n_layers, 2, 2))`, rồi `steps` bước gradient
  descent full batch `params = params - lr * grad`. Trả `(params, history)`, `history[i]` là loss **trước** bước
  thứ `i` (list dài `steps`).

In [ ]:
def loss_and_grad(model, params: np.ndarray, X: np.ndarray, y_pm: np.ndarray) -> tuple[float, np.ndarray]:
    """MSE giữa model(params, X) và y_pm, cùng gradient theo params tính bằng parameter-shift."""
    out = model(params, X)
    resid = out - y_pm
    jac = parameter_shift_jacobian(lambda p: model(p, X), params)    # shape params.shape + (B,)
    grad = jac @ (2 * resid / len(X))                                 # chain rule, cộng theo trục batch
    return float(np.mean(resid**2)), grad


def train_vqc(model, X: np.ndarray, y_pm: np.ndarray, n_layers: int, steps: int = 100, lr: float = 0.3,
              seed: int = 0) -> tuple[np.ndarray, list[float]]:
    """Gradient descent full batch từ params ~ U(−π, π) shape (n_layers, 2, 2). Trả (params, history)."""
    params = np.random.default_rng(seed).uniform(-np.pi, np.pi, size=(n_layers, 2, 2))
    history = []
    for _ in range(steps):
        loss, grad = loss_and_grad(model, params, X, y_pm)
        history.append(loss)
        params = params - lr * grad
    return params, history

## 4. Thí nghiệm trên moons

300 điểm `make_moons(noise=0.2, random_state=0)`, chia 200 train / 100 test (stratify), scale về $$[0, \pi]$$ bằng
`MinMaxScaler` fit trên train. Cell dưới train mạch với $$L = 1..4$$ bằng **code của bạn** (khoảng 40 giây tổng).

In [ ]:
# explore: dữ liệu moons, chia train/test, scale về [0, π]
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


def moons_split(seed: int):
    X, y = make_moons(n_samples=300, noise=0.2, random_state=0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=100, random_state=seed, stratify=y)
    scaler = MinMaxScaler((0, np.pi)).fit(X_tr)
    return scaler.transform(X_tr), scaler.transform(X_te), y_tr, y_te


X_tr, X_te, y_tr, y_te = moons_split(0)
y_tr_pm, y_te_pm = 2 * y_tr - 1, 2 * y_te - 1
print(X_tr.shape, X_te.shape, "tỉ lệ lớp 1 ở train:", y_tr.mean())

In [ ]:
# explore: train L = 1..4 bằng parameter-shift của bạn, 100 bước, lr 0.3 (bảng đầu mục 5 lý thuyết)
results = {}
for L in (1, 2, 3, 4):
    model = make_vqc(L)
    start = time.perf_counter()
    params, hist = train_vqc(model, X_tr, y_tr_pm, n_layers=L, steps=100, lr=0.3, seed=0)
    elapsed = time.perf_counter() - start
    acc_tr = np.mean(np.sign(model(params, X_tr)) == y_tr_pm)
    acc_te = np.mean(np.sign(model(params, X_te)) == y_te_pm)
    n_exec = 100 * (2 * params.size + 1)                 # số lần chạy QNode (mỗi lần cả batch 200 mẫu)
    results[L] = (params, hist)
    print(f"L={L}: {params.size:2d} tham số | loss {hist[0]:.3f} -> {hist[-1]:.3f} | train acc {acc_tr:.3f}"
          f" | test acc {acc_te:.3f} | {n_exec} lần chạy QNode | {elapsed:.1f} s")

In [ ]:
# explore: "light cone": tham số nào của layer cuối không ảnh hưởng <Z_0>?
model4 = make_vqc(4)
params4 = results[4][0]
_, g = loss_and_grad(model4, params4, X_tr, y_tr_pm)
print("gradient layer cuối (hàng = wire, cột = RY, RZ):")
print(np.round(g[-1], 6))
print("số tham số có gradient bằng 0 (|g| < 1e-12):", int(np.sum(np.abs(g) < 1e-12)))

In [ ]:
# explore: kiểm gradient của bạn với autograd của PennyLane (backprop qua simulator, không làm được trên phần cứng)
from pennylane import numpy as pnp


def make_vqc_backprop(n_layers):
    """QNode của bạn (model.circuit); với tham số pennylane.numpy, default.qubit lấy gradient bằng backprop."""
    return make_vqc(n_layers).circuit


circuit_bp = make_vqc_backprop(4)
p0 = np.random.default_rng(0).uniform(-np.pi, np.pi, size=(4, 2, 2))
g_bp = qml.grad(lambda p: pnp.mean((circuit_bp(p, X_tr) - y_tr_pm) ** 2))(pnp.array(p0, requires_grad=True))
_, g_ps = loss_and_grad(model4, p0, X_tr, y_tr_pm)
print("max |parameter-shift − backprop| =", float(np.max(np.abs(np.asarray(g_bp) - g_ps))))

## 5. So sánh trung thực với classical baseline

Một lần chia 100 điểm test chỉ phân biệt được chênh lệch cỡ vài điểm phần trăm: một điểm test là 1%. Cell dưới lặp lại
**5 lần chia** (seed 0..4) cho mạch $$L = 4$$ và bốn baseline của scikit-learn, cùng input đã scale. Để chạy nhanh, mạch
train bằng backprop của simulator (cell trên đã kiểm nó cho cùng gradient với parameter-shift của bạn); trên phần cứng
thật chỉ có parameter-shift.

In [ ]:
# explore: 5 lần chia, VQC L=4 vs logistic regression, SVM RBF, hai MLP (bảng thứ hai mục 5 lý thuyết)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC


def fit_vqc_backprop(X, y_pm, n_layers=4, steps=100, lr=0.3, seed=0):
    circuit = make_vqc_backprop(n_layers)
    grad_fn = qml.grad(lambda p: pnp.mean((circuit(p, X) - y_pm) ** 2))
    p = pnp.array(np.random.default_rng(seed).uniform(-np.pi, np.pi, size=(n_layers, 2, 2)), requires_grad=True)
    for _ in range(steps):
        p = p - lr * grad_fn(p)
    return lambda Xn: np.asarray(circuit(p, Xn))


rows = {name: ([], []) for name in ["VQC 2 qubit, L=4", "Logistic regression", "SVM RBF", "MLP 16 (mặc định)",
                                    "MLP 32-32"]}
for seed in range(5):
    Xa, Xb, ya, yb = moons_split(seed)
    start = time.perf_counter()
    predict = fit_vqc_backprop(Xa, 2 * ya - 1)
    rows["VQC 2 qubit, L=4"][0].append(np.mean((predict(Xb) > 0) == yb))
    rows["VQC 2 qubit, L=4"][1].append(time.perf_counter() - start)
    for name, clf in [("Logistic regression", LogisticRegression()), ("SVM RBF", SVC()),
                      ("MLP 16 (mặc định)", MLPClassifier((16,), max_iter=3000, random_state=0)),
                      ("MLP 32-32", MLPClassifier((32, 32), max_iter=3000, random_state=0))]:
        start = time.perf_counter()
        clf.fit(Xa, ya)
        rows[name][1].append(time.perf_counter() - start)
        rows[name][0].append(clf.score(Xb, yb))
for name, (accs, times) in rows.items():
    print(f"{name:20s} test acc {np.mean(accs):.3f} ± {np.std(accs):.3f} | min {np.min(accs):.2f}"
          f" | thời gian fit trung bình {np.mean(times) * 1000:8.1f} ms")

## 6. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Vì sao mạch 1 layer chỉ đạt khoảng 0.6 test accuracy? Gợi ý: với $$L = 1$$, $$x_1$$ và các gate trên wire 1 có
   ảnh hưởng gì tới $$\langle Z_0 \rangle$$ không? Liên hệ với cell "light cone".
2. Vì sao parameter-shift vẫn đúng khi đo bằng 1000 shots còn sai phân hữu hạn thì hỏng?
3. Ở lần chia seed 0, VQC $$L = 4$$ đạt 0.98 còn SVM RBF 0.96. Sau 5 lần chia thì sao? Bạn có dám viết "quantum
   tốt hơn" không? MLP 16 với siêu tham số mặc định nói gì về việc chọn baseline?
4. Trên phần cứng thật, mỗi bước train cần bao nhiêu mạch chạy (tính cả 200 mẫu và 1000 shots mỗi mạch)?